In [3]:
import jax.numpy as jnp
from jax import grad, jit, vmap
from jax import random

from potentials import dist_lin_seg
dist_lin_seg


<function potentials.dist_lin_seg(point1s, point1e, point2s, point2e)>

In [137]:
key = random.key(0)
x = random.normal(key, (10,))
print(x)

[-0.3721109   0.26423115 -0.18252768 -0.7368197  -0.44030377 -0.1521442
 -0.67135346 -0.5908641   0.73168886  0.5673026 ]


In [138]:
size = 3000
x = random.normal(key, (size, size), dtype=jnp.float32)
%timeit jnp.dot(x, x.T).block_until_ready()  # runs on the GPU

131 ms ± 7.97 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [4]:
import numpy as np
x = np.random.normal(size=(size, size)).astype(np.float32)
%timeit jnp.dot(x, x.T).block_until_ready()


134 ms ± 5.42 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [5]:
from jax import device_put

x = np.random.normal(size=(size, size)).astype(np.float32)
x = device_put(x)
%timeit jnp.dot(x, x.T).block_until_ready()

130 ms ± 5.07 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [6]:
def selu(x, alpha=1.67, lmbda=1.05):
  return lmbda * jnp.where(x > 0, x, alpha * jnp.exp(x) - alpha)

x = random.normal(key, (1000000,))
%timeit selu(x).block_until_ready()

1.45 ms ± 250 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [7]:
selu_jit = jit(selu)
%timeit selu_jit(x).block_until_ready()

390 µs ± 23.5 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [8]:
def sum_logistic(x):
  return jnp.sum(1.0 / (1.0 + jnp.exp(-x)))

x_small = jnp.arange(3.)
derivative_fn = grad(sum_logistic)
print(derivative_fn(x_small))


[0.25       0.19661197 0.10499357]


In [9]:
def first_finite_differences(f, x):
  eps = 1e-3
  return jnp.array([(f(x + eps * v) - f(x - eps * v)) / (2 * eps)
                   for v in jnp.eye(len(x))])


print(first_finite_differences(sum_logistic, x_small))

[0.24998187 0.1964569  0.10502338]


In [10]:
print(grad(jit(grad(jit(grad(sum_logistic)))))(1.0))


-0.0353256


In [11]:
from jax import jacfwd, jacrev
def hessian(fun):
  return jit(jacfwd(jacrev(fun)))

In [32]:
def f(x,y,z):
    return x**2 + y**2 + z**2

grad_f = grad(f, argnums=(0,1,2))



In [33]:
print(grad_f(1.,2.,3.))

(Array(2., dtype=float32, weak_type=True), Array(4., dtype=float32, weak_type=True), Array(6., dtype=float32, weak_type=True))


In [35]:
import jax
import jax.numpy as jnp

def f(x, y, z):
    return x**2 + y**2 + z**2

# Compute the partial derivatives
df_dx = jax.grad(f, argnums=0)  # Partial derivative with respect to x
df_dy = jax.grad(f, argnums=1)  # Partial derivative with respect to y
df_dz = jax.grad(f, argnums=2)  # Partial derivative with respect to z

# Example usage
x, y, z = 1.,2.,3.
print("df/dx:", df_dx(x, y, z))
print("df/dy:", df_dy(x, y, z))
print("df/dz:", df_dz(x, y, z))


df/dx: 2.0
df/dy: 4.0
df/dz: 6.0


In [111]:
def L(x_i, y_i, z_i, phi_i, theta_i, x_j, y_j, z_j, phi_j, theta_j, l):
    n1 = jnp.array([((z_i - z_j)*(y_j - y_i + l*jnp.sin(phi_j)*jnp.sin(theta_j)) - (y_i - y_j)*(z_j - z_i + l*jnp.cos(phi_j)))/(jnp.abs((y_i - y_j)*(x_j - x_i + l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_i - x_j)*(y_j - y_i + l*jnp.sin(phi_j)*jnp.sin(theta_j)))**2 + jnp.abs((z_i - z_j)*(x_j - x_i + l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_i - x_j)*(z_j - z_i + l*jnp.cos(phi_j)))**2 + jnp.abs((z_i - z_j)*(y_j - y_i + l*jnp.sin(phi_j)*jnp.sin(theta_j)) - (y_i - y_j)*(z_j - z_i + l*jnp.cos(phi_j)))**2)**(1/2), -((z_i - z_j)*(x_j - x_i + l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_i - x_j)*(z_j - z_i + l*jnp.cos(phi_j)))/(jnp.abs((y_i - y_j)*(x_j - x_i + l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_i - x_j)*(y_j - y_i + l*jnp.sin(phi_j)*jnp.sin(theta_j)))**2 + jnp.abs((z_i - z_j)*(x_j - x_i + l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_i - x_j)*(z_j - z_i + l*jnp.cos(phi_j)))**2 + jnp.abs((z_i - z_j)*(y_j - y_i + l*jnp.sin(phi_j)*jnp.sin(theta_j)) - (y_i - y_j)*(z_j - z_i + l*jnp.cos(phi_j)))**2)**(1/2), ((y_i - y_j)*(x_j - x_i + l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_i - x_j)*(y_j - y_i + l*jnp.sin(phi_j)*jnp.sin(theta_j)))/(jnp.abs((y_i - y_j)*(x_j - x_i + l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_i - x_j)*(y_j - y_i + l*jnp.sin(phi_j)*jnp.sin(theta_j)))**2 + jnp.abs((z_i - z_j)*(x_j - x_i + l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_i - x_j)*(z_j - z_i + l*jnp.cos(phi_j)))**2 + jnp.abs((z_i - z_j)*(y_j - y_i + l*jnp.sin(phi_j)*jnp.sin(theta_j)) - (y_i - y_j)*(z_j - z_i + l*jnp.cos(phi_j)))**2)**(1/2)])
    n2 = jnp.array([((z_j - z_i + l*jnp.cos(phi_j))*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i) - l*jnp.sin(phi_j)*jnp.sin(theta_j)) - (y_j - y_i + l*jnp.sin(phi_j)*jnp.sin(theta_j))*(z_i - z_j + l*jnp.cos(phi_i) - l*jnp.cos(phi_j)))/(jnp.abs((z_j - z_i + l*jnp.cos(phi_j))*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i) - l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_j - x_i + l*jnp.cos(theta_j)*jnp.sin(phi_j))*(z_i - z_j + l*jnp.cos(phi_i) - l*jnp.cos(phi_j)))**2 + jnp.abs((z_j - z_i + l*jnp.cos(phi_j))*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i) - l*jnp.sin(phi_j)*jnp.sin(theta_j)) - (y_j - y_i + l*jnp.sin(phi_j)*jnp.sin(theta_j))*(z_i - z_j + l*jnp.cos(phi_i) - l*jnp.cos(phi_j)))**2 + jnp.abs((y_j - y_i + l*jnp.sin(phi_j)*jnp.sin(theta_j))*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i) - l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_j - x_i + l*jnp.cos(theta_j)*jnp.sin(phi_j))*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i) - l*jnp.sin(phi_j)*jnp.sin(theta_j)))**2)**(1/2), -((z_j - z_i + l*jnp.cos(phi_j))*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i) - l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_j - x_i + l*jnp.cos(theta_j)*jnp.sin(phi_j))*(z_i - z_j + l*jnp.cos(phi_i) - l*jnp.cos(phi_j)))/(jnp.abs((z_j - z_i + l*jnp.cos(phi_j))*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i) - l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_j - x_i + l*jnp.cos(theta_j)*jnp.sin(phi_j))*(z_i - z_j + l*jnp.cos(phi_i) - l*jnp.cos(phi_j)))**2 + jnp.abs((z_j - z_i + l*jnp.cos(phi_j))*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i) - l*jnp.sin(phi_j)*jnp.sin(theta_j)) - (y_j - y_i + l*jnp.sin(phi_j)*jnp.sin(theta_j))*(z_i - z_j + l*jnp.cos(phi_i) - l*jnp.cos(phi_j)))**2 + jnp.abs((y_j - y_i + l*jnp.sin(phi_j)*jnp.sin(theta_j))*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i) - l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_j - x_i + l*jnp.cos(theta_j)*jnp.sin(phi_j))*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i) - l*jnp.sin(phi_j)*jnp.sin(theta_j)))**2)**(1/2), ((y_j - y_i + l*jnp.sin(phi_j)*jnp.sin(theta_j))*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i) - l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_j - x_i + l*jnp.cos(theta_j)*jnp.sin(phi_j))*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i) - l*jnp.sin(phi_j)*jnp.sin(theta_j)))/(jnp.abs((z_j - z_i + l*jnp.cos(phi_j))*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i) - l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_j - x_i + l*jnp.cos(theta_j)*jnp.sin(phi_j))*(z_i - z_j + l*jnp.cos(phi_i) - l*jnp.cos(phi_j)))**2 + jnp.abs((z_j - z_i + l*jnp.cos(phi_j))*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i) - l*jnp.sin(phi_j)*jnp.sin(theta_j)) - (y_j - y_i + l*jnp.sin(phi_j)*jnp.sin(theta_j))*(z_i - z_j + l*jnp.cos(phi_i) - l*jnp.cos(phi_j)))**2 + jnp.abs((y_j - y_i + l*jnp.sin(phi_j)*jnp.sin(theta_j))*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i) - l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_j - x_i + l*jnp.cos(theta_j)*jnp.sin(phi_j))*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i) - l*jnp.sin(phi_j)*jnp.sin(theta_j)))**2)**(1/2)]) 
    n3 = jnp.array([((z_i - z_j + l*jnp.cos(phi_i))*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i) - l*jnp.sin(phi_j)*jnp.sin(theta_j)) - (y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i))*(z_i - z_j + l*jnp.cos(phi_i) - l*jnp.cos(phi_j)))/(jnp.abs((y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i))*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i) - l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i))*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i) - l*jnp.sin(phi_j)*jnp.sin(theta_j)))**2 + jnp.abs((z_i - z_j + l*jnp.cos(phi_i))*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i) - l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i))*(z_i - z_j + l*jnp.cos(phi_i) - l*jnp.cos(phi_j)))**2 + jnp.abs((z_i - z_j + l*jnp.cos(phi_i))*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i) - l*jnp.sin(phi_j)*jnp.sin(theta_j)) - (y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i))*(z_i - z_j + l*jnp.cos(phi_i) - l*jnp.cos(phi_j)))**2)**(1/2), -((z_i - z_j + l*jnp.cos(phi_i))*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i) - l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i))*(z_i - z_j + l*jnp.cos(phi_i) - l*jnp.cos(phi_j)))/(jnp.abs((y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i))*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i) - l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i))*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i) - l*jnp.sin(phi_j)*jnp.sin(theta_j)))**2 + jnp.abs((z_i - z_j + l*jnp.cos(phi_i))*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i) - l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i))*(z_i - z_j + l*jnp.cos(phi_i) - l*jnp.cos(phi_j)))**2 + jnp.abs((z_i - z_j + l*jnp.cos(phi_i))*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i) - l*jnp.sin(phi_j)*jnp.sin(theta_j)) - (y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i))*(z_i - z_j + l*jnp.cos(phi_i) - l*jnp.cos(phi_j)))**2)**(1/2), ((y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i))*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i) - l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i))*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i) - l*jnp.sin(phi_j)*jnp.sin(theta_j)))/(jnp.abs((y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i))*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i) - l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i))*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i) - l*jnp.sin(phi_j)*jnp.sin(theta_j)))**2 + jnp.abs((z_i - z_j + l*jnp.cos(phi_i))*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i) - l*jnp.cos(theta_j)*jnp.sin(phi_j)) - (x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i))*(z_i - z_j + l*jnp.cos(phi_i) - l*jnp.cos(phi_j)))**2 + jnp.abs((z_i - z_j + l*jnp.cos(phi_i))*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i) - l*jnp.sin(phi_j)*jnp.sin(theta_j)) - (y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i))*(z_i - z_j + l*jnp.cos(phi_i) - l*jnp.cos(phi_j)))**2)**(1/2)])
    n4 = jnp.array([((z_i - z_j)*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i)) - (y_i - y_j)*(z_i - z_j + l*jnp.cos(phi_i)))/(jnp.abs((y_i - y_j)*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i)) - (x_i - x_j)*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i)))**2 + jnp.abs((z_i - z_j)*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i)) - (x_i - x_j)*(z_i - z_j + l*jnp.cos(phi_i)))**2 + jnp.abs((z_i - z_j)*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i)) - (y_i - y_j)*(z_i - z_j + l*jnp.cos(phi_i)))**2)**(1/2), -((z_i - z_j)*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i)) - (x_i - x_j)*(z_i - z_j + l*jnp.cos(phi_i)))/(jnp.abs((y_i - y_j)*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i)) - (x_i - x_j)*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i)))**2 + jnp.abs((z_i - z_j)*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i)) - (x_i - x_j)*(z_i - z_j + l*jnp.cos(phi_i)))**2 + jnp.abs((z_i - z_j)*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i)) - (y_i - y_j)*(z_i - z_j + l*jnp.cos(phi_i)))**2)**(1/2), ((y_i - y_j)*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i)) - (x_i - x_j)*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i)))/(jnp.abs((y_i - y_j)*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i)) - (x_i - x_j)*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i)))**2 + jnp.abs((z_i - z_j)*(x_i - x_j + l*jnp.cos(theta_i)*jnp.sin(phi_i)) - (x_i - x_j)*(z_i - z_j + l*jnp.cos(phi_i)))**2 + jnp.abs((z_i - z_j)*(y_i - y_j + l*jnp.sin(phi_i)*jnp.sin(theta_i)) - (y_i - y_j)*(z_i - z_j + l*jnp.cos(phi_i)))**2)**(1/2)])

    return 1/4/jnp.pi*(jnp.arcsin(jnp.dot(n1,n2)) + jnp.arcsin(jnp.dot(n2,n3)) + jnp.arcsin(jnp.dot(n3,n4)) + jnp.arcsin(jnp.dot(n4,n1)))

rod_length = 10.
rod_diameter = 1.
dist = rod_diameter/2

L(-rod_length/2,0,0,
  jnp.pi/2,0,
  0,-rod_length/2,dist,
  jnp.pi/2,jnp.pi/2,
  rod_length)

# lol, normalization

Array(0.45517054, dtype=float32)

In [112]:
grad_L = jax.grad(L, argnums=(0,1,2,3,4,5,6,7,8,9,10))

tmp = grad_L(-rod_length/2,0.,0.,
    np.pi/2,0.,
    0.,-rod_length/2,dist,
    np.pi/2,np.pi/2,
    rod_length)

print(tmp)

(Array(-9.009893e-10, dtype=float32, weak_type=True), Array(7.4505815e-09, dtype=float32, weak_type=True), Array(0.08891821, dtype=float32, weak_type=True), Array(-0.44459102, dtype=float32, weak_type=True), Array(-1.2016675e-07, dtype=float32, weak_type=True), Array(9.009893e-10, dtype=float32, weak_type=True), Array(-7.4505815e-09, dtype=float32, weak_type=True), Array(-0.08891821, dtype=float32, weak_type=True), Array(0.44459105, dtype=float32, weak_type=True), Array(-8.953107e-09, dtype=float32, weak_type=True), Array(0.00444598, dtype=float32, weak_type=True))


In [113]:
L_x = jax.grad(L,argnums=2)
print(L_x(-rod_length/2,1.,0.,
    np.pi/2,0.,
    0.,-rod_length/2,dist,
    np.pi/2,np.pi/2,
    rod_length))
# def L_vectorized(x):
#     return L(*x)


0.09111861


In [114]:
def L2(x_i, y_i, z_i, phi_i, theta_i, x_j, y_j, z_j, phi_j, theta_j, l):
    p_i = jnp.array([x_i, y_i, z_i])
    p_j = jnp.array([x_j, y_j, z_j])
    u_i = jnp.array([jnp.sin(phi_i)*jnp.cos(theta_i), jnp.sin(phi_i)*jnp.sin(theta_i), jnp.cos(phi_i)])
    u_j = jnp.array([jnp.sin(phi_j)*jnp.cos(theta_j), jnp.sin(phi_j)*jnp.sin(theta_j), jnp.cos(phi_j)])

    p_ii = p_i + l*u_i
    p_jj = p_j + l*u_j

    r_ij = p_i - p_j
    r_ijj = p_i - p_jj
    r_iij = p_ii - p_j
    r_iijj = p_ii - p_jj

    n1 = jnp.cross(r_ij, r_ijj)
    n1 = n1/jnp.linalg.norm(n1)
    n2 = jnp.cross(r_ijj, r_iijj)
    n2 = n2/jnp.linalg.norm(n2)
    n3 = jnp.cross(r_iijj, r_iij)
    n3 = n3/jnp.linalg.norm(n3)
    n4 = jnp.cross(r_iij, r_ij)
    n4 = n4/jnp.linalg.norm(n4)

    return 1/4/jnp.pi*(jnp.arcsin(jnp.dot(n1,n2)) + jnp.arcsin(jnp.dot(n2,n3)) + jnp.arcsin(jnp.dot(n3,n4)) + jnp.arcsin(jnp.dot(n4,n1)))

In [128]:
# l1 = L(-rod_length/2,0,0,jnp.pi/2,0,0,-rod_length/2,dist,jnp.pi/2,jnp.pi/2,rod_length)
%timeit L(-rod_length/2,0,0,jnp.pi/2,0,0,-rod_length/2,dist,jnp.pi/2,jnp.pi/2,rod_length)
%timeit L2(-rod_length/2,0,0,jnp.pi/2,0,0,-rod_length/2,dist,jnp.pi/2,jnp.pi/2,rod_length)
# l2 = L2(-rod_length/2,0,0,
#   jnp.pi/2,0,
#   0,-rod_length/2,dist,
#   jnp.pi/2,jnp.pi/2,
#   rod_length)
# %timeit jnp.dot(x, x.T).block_until_ready()
        

# print(l1,l2)

6.2 ms ± 74.5 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
820 µs ± 9.24 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [125]:
L_x = jax.grad(L2,argnums=(0,1,2,3,4,5,6,7,8,9,10))
print(L_x(-rod_length/2,1.,0.,
    np.pi/2,0.,
    0.,-rod_length/2,dist,
    np.pi/2,np.pi/2,
    rod_length))


(Array(2.0838343e-08, dtype=float32, weak_type=True), Array(-0.00240987, dtype=float32, weak_type=True), Array(0.09111861, dtype=float32, weak_type=True), Array(-0.455593, dtype=float32, weak_type=True), Array(-0.01204935, dtype=float32, weak_type=True), Array(-2.0838343e-08, dtype=float32, weak_type=True), Array(0.00240987, dtype=float32, weak_type=True), Array(-0.09111861, dtype=float32, weak_type=True), Array(0.5455067, dtype=float32, weak_type=True), Array(1.2289797e-07, dtype=float32, weak_type=True), Array(0.00600181, dtype=float32, weak_type=True))


In [133]:
import jax.numpy as jnp
from jax import lax

def factorial(n):
  """Computes the factorial of n."""
  count = jnp.int32(0)
  acc = jnp.int32(1)
  def cond_fun(carry):
    count, acc = carry
    return count < n
  def body_fun(carry):
    count, acc = carry
    return count + 1, acc * count
  carry = (count, acc)
  carry = lax.while_loop(cond_fun, body_fun, carry)
  return carry[1]

print(factorial(1))

0


In [134]:
import jax
import jax.numpy as jnp
from jax.lax import while_loop

def factorial(n):
    # Initial state: (current result, current number)
    initial_state = (1, 1)

    # The condition function: run while i <= n
    def cond_fun(state):
        _, i = state
        return i <= n

    # The body function: update the result and increment i
    def body_fun(state):
        result, i = state
        return result * i, i + 1

    # Run the while loop
    final_result, _ = while_loop(cond_fun, body_fun, initial_state)
    return final_result

# Example usage:
n = 5
result = factorial(n)
print("Factorial of", n, "is", result)


Factorial of 5 is 120


In [8]:
from potentials import dist_lin_seg
p1s = jnp.array([0, -10, 0])
p1e = jnp.array([0, 10, 0])
p2s = jnp.array([0, -10, 5.2323423])
p2e = jnp.array([0, 10, 5.234234])

dist = dist_lin_seg(p1s, p1e, p2s, p2e)
# print(dist)
%timeit dist_lin_seg(p1s, p1e, p2s, p2e)



137 µs ± 1.12 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


SyntaxError: invalid syntax (potentials.py, line 126)

In [15]:
x = 0
y = 0
f = lambda a : a + 10

f(5)

15